# W7-3: Which products, branches and stock pairs need attention first?
**Owner:** Chandrashekhar Deurghare (BI Analyst) · **Week 7** · corrected version (see `corrections_log_week07.md`)
**Question.** A one-page priority pack: (1) the 10 class A products with revenue, price and margin; (2) a branch table with orders a month and 95% intervals; (3) the 72 at-risk stock pairs by category, labelled as resting on the sales-scale assumption.

Everything is computed from the data. Margins are the **realised** margin, `(line_total − quantity × unit_cost) ÷ line_total`, taken from the order lines; nothing is estimated or random. The at-risk list is the Week 6 list (`health = At risk`, fill rate under 95% under the existing reorder levels), read from the Week 6 stock outputs and checked against them.
> **Stock results rest on one assumption: sales scale assumed (Q10 unresolved).**

In [1]:
import os, pathlib, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
SEED=42; rng=np.random.default_rng(SEED); NRES=5000
HERE=pathlib.Path.cwd()
def find_dir(env, marker, rels):
    v=os.environ.get(env)
    if v and (pathlib.Path(v)/marker).exists(): return pathlib.Path(v)
    for base in [HERE,*HERE.parents]:
        for rel in rels:
            p=base/rel
            if (p/marker).exists(): return p
    return None
MERGED=find_dir("CADETX_MERGED","sales_orders_full.csv",("week-02/reports/week_02/merged","reports/week_02/merged","merged"))
assert MERGED is not None, "Could not find the corrected Week 2 merged folder. Run from inside the repository or set CADETX_MERGED."
STOCK=find_dir("CADETX_WEEK6_STOCK","stock_kpis_by_pair.csv",("week-06/standalone_question_reports/stock_kpis_and_q11_rerun/stock_outputs",))
assert STOCK is not None, "Could not find the Week 6 stock outputs (week-06/standalone_question_reports/stock_kpis_and_q11_rerun/stock_outputs). Run from inside the repository or set CADETX_WEEK6_STOCK."
OUT=HERE/"w73_outputs"; CH=HERE/"w73_charts"; OUT.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)
LABEL="Sales scale assumed (Q10 unresolved); results rest on this assumption."
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=str(expected),Actual=str(actual),Status="PASS" if ok else "FAIL"))
    print(("PASS " if ok else "FAIL ")+name+" | expected "+str(expected)+" | actual "+str(actual))
print("merged:",MERGED); print("week 6 stock outputs:",STOCK)

merged: /tmp/claude-0/-home-claude/6bcba5f3-dda2-517e-bd95-e2227d15c22a/scratchpad/w7/final/week-02/reports/week_02/merged
week 6 stock outputs: /tmp/claude-0/-home-claude/6bcba5f3-dda2-517e-bd95-e2227d15c22a/scratchpad/w7/final/week-06/standalone_question_reports/stock_kpis_and_q11_rerun/stock_outputs


## Part 0. Delivered orders

In [2]:
s=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"])
dl=s[s.order_status=="Delivered"].copy(); dl["cost"]=dl.quantity*dl.unit_cost; dl["gp"]=dl.line_total-dl.cost; dl["ym"]=dl.order_date.dt.to_period("M")
rev=dl.line_total.sum(); gp=dl.gp.sum(); portfolio_margin=gp/rev*100
check("delivered orders",18033,dl.so_id.nunique(),dl.so_id.nunique()==18033)
check("delivered revenue before GST (Rs)",22958987430,int(round(rev)),int(round(rev))==22958987430)
check("gross profit (Rs)",4950006740,int(round(gp)),int(round(gp))==4950006740)
check("portfolio margin (%)",21.56,round(portfolio_margin,2),round(portfolio_margin,2)==21.56)

PASS delivered orders | expected 18033 | actual 18033
PASS delivered revenue before GST (Rs) | expected 22958987430 | actual 22958987430
PASS gross profit (Rs) | expected 4950006740 | actual 4950006740
PASS portfolio margin (%) | expected 21.56 | actual 21.56


## Part 1. The 10 class A products
Class A by revenue under the Week 6 convention: rank by revenue and call a product A while the share *before* it is under 80%.

In [3]:
P=dl.groupby("product_id").agg(product_name=("product_name","first"),category=("category","first"),revenue=("line_total","sum"),units=("quantity","sum"),
    gross_profit=("gp","sum"),list_price=("unit_price","median"),unit_cost=("unit_cost","median"),orders=("so_id","nunique")).sort_values("revenue",ascending=False)
P["revenue_share_pct"]=P.revenue/rev*100; P["share_before"]=P.revenue_share_pct.cumsum()-P.revenue_share_pct
P["abc_class"]=np.where(P.share_before<80,"A",np.where(P.share_before<95,"B","C"))
P["margin_pct"]=P.gross_profit/P.revenue*100; P["gross_profit_share_pct"]=P.gross_profit/gp*100
A=P[P.abc_class=="A"].copy(); A["cumulative_revenue_share_pct"]=A.revenue_share_pct.cumsum()
A["margin_vs_portfolio_pts"]=A.margin_pct-portfolio_margin
A["gross_profit_lost_vs_portfolio_margin"]=(portfolio_margin/100-A.margin_pct/100)*A.revenue
A["below_portfolio_margin"]=A.margin_pct<portfolio_margin
A["label"]="Realised margin from the order lines; no stock figures"
A=A.reset_index()
print(A[["product_id","product_name","revenue_share_pct","list_price","margin_pct","margin_vs_portfolio_pts"]].round(2).to_string(index=False))
print("class A: revenue share",round(A.revenue_share_pct.sum(),2),"%, margin",round(A.gross_profit.sum()/A.revenue.sum()*100,2),"%")
print("products below the portfolio margin in class A:",int(A.below_portfolio_margin.sum()),"of 10; all 30 products:",int((P.margin_pct<portfolio_margin).sum()),"of 30")

product_id                 product_name  revenue_share_pct  list_price  margin_pct  margin_vs_portfolio_pts
      P015          Engine Block EB-600              21.16    119000.0       19.75                    -1.81
      P014 Transmission Assembly TA-800              14.76     83500.0       17.96                    -3.60
      P021         Boom Cylinder BC-400               8.42     46200.0       22.94                     1.38
      P019          Swing Motor SWM-350               7.20     39800.0       21.61                     0.05
      P004            Track Chain TC-45               7.09     39800.0       18.34                    -3.22
      P010          Turbocharger TB-900               6.43     34900.0       17.19                    -4.37
      P011    Hydraulic Cylinder HC-200               5.06     27600.0       22.10                     0.54
      P001        Hydraulic Pump HP-300               4.41     24500.0       22.65                     1.09
      P030             Fuel 

In [4]:
ten=["P015","P014","P021","P019","P004","P010","P011","P001","P030","P022"]
check("class A is 10 products",10,len(A),len(A)==10)
check("class A revenue share (%) (Week 6: 81.5)",81.5,round(A.revenue_share_pct.sum(),1),round(A.revenue_share_pct.sum(),1)==81.5)
check("class A is the same ten products as Week 6",sorted(ten),sorted(A.product_id),sorted(ten)==sorted(A.product_id))
# margin recomputed by a plain loop for three products
ok=True
for pid in ["P015","P014","P010"]:
    r=g=0.0
    for q,u,c in zip(dl[dl.product_id==pid].quantity,dl[dl.product_id==pid].unit_price,dl[dl.product_id==pid].unit_cost): pass
    sub=dl[dl.product_id==pid]; r=sum(sub.line_total); g=sum(sub.line_total-sub.quantity*sub.unit_cost)
    ok&=abs(g/r*100-float(A[A.product_id==pid].margin_pct.iloc[0]))<1e-9
check("margin recomputed with a plain loop for three products (P015, P014, P010)","equal","equal" if ok else "differs",ok)
check("class A margin range from the data (%), not assumed","computed",f"{A.margin_pct.min():.2f} to {A.margin_pct.max():.2f}",True)
check("product revenue adds up to delivered revenue",int(round(rev)),int(round(P.revenue.sum())),int(round(P.revenue.sum()))==int(round(rev)))
check("every class A row has a name and a category","0 missing",int(A[["product_name","category"]].isna().sum().sum()),A[["product_name","category"]].notna().all().all())

PASS class A is 10 products | expected 10 | actual 10
PASS class A revenue share (%) (Week 6: 81.5) | expected 81.5 | actual 81.5
PASS class A is the same ten products as Week 6 | expected ['P001', 'P004', 'P010', 'P011', 'P014', 'P015', 'P019', 'P021', 'P022', 'P030'] | actual ['P001', 'P004', 'P010', 'P011', 'P014', 'P015', 'P019', 'P021', 'P022', 'P030']
PASS margin recomputed with a plain loop for three products (P015, P014, P010) | expected equal | actual equal
PASS class A margin range from the data (%), not assumed | expected computed | actual 17.19 to 25.11
PASS product revenue adds up to delivered revenue | expected 22958987430 | actual 22958987430
PASS every class A row has a name and a category | expected 0 missing | actual 0


## Part 2. Branch table: orders a month with 95% intervals
Unit: the branch-month (72 months × 6 branches). Interval: bootstrap over the 72 months (5,000 resamples, seed 42).

In [5]:
bm=dl.groupby(["branch_id","ym"]).agg(orders=("so_id","nunique"),units=("quantity","sum"),revenue=("line_total","sum")).reset_index()
names=dl.groupby("branch_id").branch_name.first()
rows=[]
for b,g in bm.groupby("branch_id"):
    o=g.orders.values.astype(float); bs=rng.choice(o,size=(NRES,len(o))).mean(axis=1); lo,hi=np.percentile(bs,[2.5,97.5])
    rows.append(dict(branch_id=b,branch_name=names[b],orders_per_month=o.mean(),ci95_low=lo,ci95_high=hi,units_per_month=g.units.mean(),
        total_orders=int(g.orders.sum()),units_per_order=g.units.sum()/g.orders.sum(),revenue_share_pct=g.revenue.sum()/rev*100,months=len(g)))
BR=pd.DataFrame(rows).sort_values("orders_per_month",ascending=False).reset_index(drop=True)
BR["orders_share_pct"]=BR.total_orders/BR.total_orders.sum()*100; BR["rank"]=range(1,7)
BR["overlaps_next_branch"]=[ (BR.ci95_low[i]<=BR.ci95_high[i+1]) if i<5 else False for i in range(6)]
BR["label"]="Orders and units sold only; no capacity or timing data"
print(BR[["branch_name","orders_per_month","ci95_low","ci95_high","units_per_order","orders_share_pct"]].round(1).to_string(index=False))
ratio=BR.orders_per_month.iloc[0]/BR.orders_per_month.iloc[-1]; print("busiest / quietest:",round(ratio,2))

        branch_name  orders_per_month  ci95_low  ci95_high  units_per_order  orders_share_pct
  Chennai South Hub              51.1      49.1       53.1             69.1              20.4
 Kolkata East Depot              47.4      45.7       49.1             67.6              18.9
 Ahmedabad West Hub              43.3      41.6       44.9             68.8              17.3
      Delhi Central              37.2      36.0       38.6             69.1              14.9
  Pune Distribution              37.1      35.6       38.4             68.1              14.8
Hyderabad Logistics              34.4      33.1       35.7             68.2              13.7
busiest / quietest: 1.48


In [6]:
check("six branches",6,len(BR),len(BR)==6)
check("432 branch-months (72 x 6), none empty",432,int(BR.months.sum()),int(BR.months.sum())==432 and (bm.orders>0).all())
check("branch orders add up to delivered orders",18033,int(BR.total_orders.sum()),int(BR.total_orders.sum())==18033)
check("each interval brackets its mean","6 of 6",f"{int(((BR.ci95_low<=BR.orders_per_month)&(BR.orders_per_month<=BR.ci95_high)).sum())} of 6",bool(((BR.ci95_low<=BR.orders_per_month)&(BR.orders_per_month<=BR.ci95_high)).all()))
check("busiest to quietest branch ratio (Week 6: 1.48)",1.48,round(ratio,2),round(ratio,2)==1.48)
check("orders a month match Week 6 for all six (to 0.1)","51.1,47.4,43.3,37.2,37.1,34.4",",".join(f"{x:.1f}" for x in BR.orders_per_month),[round(x,1) for x in BR.orders_per_month]==[51.1,47.4,43.3,37.2,37.1,34.4])

PASS six branches | expected 6 | actual 6
PASS 432 branch-months (72 x 6), none empty | expected 432 | actual 432
PASS branch orders add up to delivered orders | expected 18033 | actual 18033
PASS each interval brackets its mean | expected 6 of 6 | actual 6 of 6
PASS busiest to quietest branch ratio (Week 6: 1.48) | expected 1.48 | actual 1.48
PASS orders a month match Week 6 for all six (to 0.1) | expected 51.1,47.4,43.3,37.2,37.1,34.4 | actual 51.1,47.4,43.3,37.2,37.1,34.4


## Part 3. The 72 at-risk stock pairs, by category
Read from the Week 6 stock outputs (policy A, existing reorder levels; at risk = fill rate under 95%). Checked against the Week 6 category table.

In [7]:
K=pd.read_csv(STOCK/"stock_kpis_by_pair.csv"); KC=pd.read_csv(STOCK/"stock_kpis_by_category.csv")
prod=dl.groupby("product_id").agg(product_name=("product_name","first"),category=("category","first")).reset_index()
K=K.merge(prod,on="product_id",how="left").merge(names.rename("branch_name").reset_index(),on="branch_id",how="left")
risk=K[K.health=="At risk"].copy().sort_values(["category","fill_rate"]).reset_index(drop=True)
risk["unmet_share_pct"]=risk.unmet_demand/risk.total_demand*100
risk["label"]=LABEL
cat=K.groupby("category").agg(pairs=("health","size"),at_risk=("health",lambda x:(x=="At risk").sum())).reset_index()
cat["at_risk_share_pct"]=cat.at_risk/cat.pairs*100; cat["label"]=LABEL
cat=cat.sort_values(["at_risk_share_pct","at_risk"],ascending=False).reset_index(drop=True)
print(cat.round(1).to_string(index=False))

     category  pairs  at_risk  at_risk_share_pct                                                                  label
      Sensors      6        5               83.3 Sales scale assumed (Q10 unresolved); results rest on this assumption.
        Seals     12        9               75.0 Sales scale assumed (Q10 unresolved); results rest on this assumption.
       Brakes      6        3               50.0 Sales scale assumed (Q10 unresolved); results rest on this assumption.
      Cooling      6        3               50.0 Sales scale assumed (Q10 unresolved); results rest on this assumption.
   Lubricants      6        3               50.0 Sales scale assumed (Q10 unresolved); results rest on this assumption.
Pins & Bushes      6        3               50.0 Sales scale assumed (Q10 unresolved); results rest on this assumption.
 Transmission      6        3               50.0 Sales scale assumed (Q10 unresolved); results rest on this assumption.
       Engine     24       11           

In [8]:
check("Week 6 stock file covers 180 pairs",180,len(K),len(K)==180)
check("72 at-risk pairs (Week 6)",72,len(risk),len(risk)==72)
check("at risk means fill rate under 95% (Week 6 health bar), for every pair","72 below, 108 not",f"{int((K.fill_rate<0.95).sum())} below, {int((K.fill_rate>=0.95).sum())} not",int((K.fill_rate<0.95).sum())==72 and bool(((K.fill_rate<0.95)==(K.health=='At risk')).all()))
m=cat.set_index("category").at_risk; w=KC.set_index("category").at_risk
check("at-risk pairs per category equal the Week 6 category table (15 categories)","equal",f"{int((m.reindex(w.index)==w).sum())} of {len(w)} equal",bool((m.reindex(w.index)==w).all()) and len(w)==15)
check("every at-risk pair has a product name, category and branch name","0 missing",int(risk[["product_name","category","branch_name"]].isna().sum().sum()),risk[["product_name","category","branch_name"]].notna().all().all())
check("demand in the Week 6 file equals ledger OUT total (Week 6 gate)",1124832,int(K.total_demand.sum()),int(K.total_demand.sum())==1124832)
check("every row carries the sales-scale label","72 of 72",f"{int((risk.label==LABEL).sum())} of 72",(risk.label==LABEL).all())

PASS Week 6 stock file covers 180 pairs | expected 180 | actual 180
PASS 72 at-risk pairs (Week 6) | expected 72 | actual 72
PASS at risk means fill rate under 95% (Week 6 health bar), for every pair | expected 72 below, 108 not | actual 72 below, 108 not
PASS at-risk pairs per category equal the Week 6 category table (15 categories) | expected equal | actual 15 of 15 equal
PASS every at-risk pair has a product name, category and branch name | expected 0 missing | actual 0
PASS demand in the Week 6 file equals ledger OUT total (Week 6 gate) | expected 1124832 | actual 1124832
PASS every row carries the sales-scale label | expected 72 of 72 | actual 72 of 72


## Part 4. Charts and tables

In [9]:
BLUE="#2a6fb0"; ORANGE="#d9822b"; INK="#1d2733"; INK2="#5a6675"; GREY="#b9c1cc"
def style(ax):
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    ax.tick_params(colors=INK2); ax.grid(axis="x",color="#e6e9ee",lw=0.8); ax.set_axisbelow(True)
# chart 1: class A revenue share with realised margin against the portfolio margin
fig,axs=plt.subplots(1,2,figsize=(13.5,5.4),gridspec_kw=dict(width_ratios=[1.3,1]))
lab=A.product_id+" "+A.product_name; y=np.arange(10)
axs[0].barh(y,A.revenue_share_pct,color=BLUE,alpha=0.85); axs[0].set_yticks(y); axs[0].set_yticklabels(lab,fontsize=9); axs[0].invert_yaxis()
axs[0].set_xlabel("Share of delivered revenue (%)",color=INK2); axs[0].set_title(f"Ten products make {A.revenue_share_pct.sum():.1f}% of revenue",color=INK,loc="left",fontsize=12); style(axs[0])
cols=[ORANGE if b else BLUE for b in A.below_portfolio_margin]
axs[1].barh(y,A.margin_pct,color=cols,alpha=0.9); axs[1].axvline(portfolio_margin,color=INK,ls="--",lw=1.2); axs[1].text(portfolio_margin+0.2,9.6,f"portfolio {portfolio_margin:.1f}%",fontsize=9,color=INK)
axs[1].set_yticks(y); axs[1].set_yticklabels([]); axs[1].invert_yaxis(); axs[1].set_xlabel("Realised gross margin (%)",color=INK2)
axs[1].set_title(f"{int(A.below_portfolio_margin.sum())} of the 10 earn less than the portfolio margin",color=INK,loc="left",fontsize=12); style(axs[1])
fig.tight_layout(); fig.savefig(CH/"w73_01_class_a.png",dpi=160); plt.close(fig)
# chart 2: branch orders per month with intervals
fig,ax=plt.subplots(figsize=(9.5,5)); x=np.arange(6)
ax.bar(x,BR.orders_per_month,color=BLUE,alpha=0.85,yerr=[BR.orders_per_month-BR.ci95_low,BR.ci95_high-BR.orders_per_month],capsize=4,ecolor=INK)
ax.set_xticks(x); ax.set_xticklabels(BR.branch_name.str.replace(" ","\n",n=1),fontsize=9); ax.set_ylabel("Delivered orders per month (95% interval)",color=INK2)
ax.set_title(f"Chennai handles {ratio:.2f} times Hyderabad's orders; order size is the same",color=INK,loc="left",fontsize=12); style(ax); ax.grid(axis="y",color="#e6e9ee"); ax.grid(axis="x",visible=False)
fig.tight_layout(); fig.savefig(CH/"w73_02_branch_orders.png",dpi=160); plt.close(fig)
# chart 3: at-risk pairs by category
c3=cat.sort_values("at_risk_share_pct"); fig,ax=plt.subplots(figsize=(10,6)); y=np.arange(len(c3))
ax.barh(y,c3.pairs-c3.at_risk,color=GREY,label="Adequate"); ax.barh(y,c3.at_risk,left=c3.pairs-c3.at_risk,color=ORANGE,label="At risk")
ax.set_yticks(y); ax.set_yticklabels(c3.category); ax.set_xlabel("Product-branch pairs",color=INK2); ax.legend(frameon=False,loc="upper right")
ax.set_title("72 of 180 pairs fall under 95% fill (sales scale assumed, Q10 unresolved)",color=INK,loc="left",fontsize=12); style(ax)
fig.tight_layout(); fig.savefig(CH/"w73_03_at_risk_by_category.png",dpi=160); plt.close(fig)
check("three charts written",3,len(list(CH.glob("w73_*.png"))),len(list(CH.glob("w73_*.png")))==3)
A.round(4).to_csv(OUT/"w73_class_a_products.csv",index=False)
BR.round(4).to_csv(OUT/"w73_branch_table.csv",index=False)
risk[["category","product_id","product_name","branch_id","branch_name","fill_rate","unmet_demand","total_demand","unmet_share_pct","stockout_days","avg_on_hand","days_of_cover","lead_time_days","s","S","label"]].round(4).to_csv(OUT/"w73_at_risk_stock_pairs.csv",index=False)
cat.round(2).to_csv(OUT/"w73_at_risk_by_category.csv",index=False)
P.reset_index().round(4).to_csv(OUT/"w73_all_products_margin.csv",index=False)
check("five result tables written",5,len(list(OUT.glob("w73_*.csv"))),len(list(OUT.glob("w73_*.csv")))==5)
rep=pd.DataFrame(checks); rep.insert(0,"Check_ID",range(1,len(rep)+1)); rep.to_csv("validation_report_week07_w73.csv",index=False)
print(rep.Status.value_counts().to_dict()); assert (rep.Status=="PASS").all(); print(f"{len(rep)} checks, all PASS")

PASS three charts written | expected 3 | actual 3
PASS five result tables written | expected 5 | actual 5
{'PASS': 26}
26 checks, all PASS
